# JAX-BO quickstart

A small CPU-only Bayesian optimization loop using the 0.2.0 core API.


In [ ]:
import jax
jax.config.update("jax_platform_name", "cpu")

import jax.numpy as jnp
from jax import random

from jaxbo import acquisitions
from jaxbo.gp import GP
from jaxbo.priors import uniform_prior
from jaxbo.utils import normalize


## Define the objective and observations


In [ ]:
bounds = {"lb": jnp.array([-2.0]), "ub": jnp.array([2.0])}

def objective(x):
    return (x - 0.65) ** 2 + 0.05 * jnp.sin(4.0 * x)

X = jnp.array([[-1.8], [-0.7], [0.4], [1.5]])
y = objective(X)
batch, norm_const = normalize(X, y, bounds)
batch["y"].shape


## Fit a Gaussian process

`GP.train` consumes the normalized batch. Predictions and candidate scoring use raw domain coordinates.


In [ ]:
model = GP({
    "kernel": "RBF",
    "criterion": "EI",
    "input_prior": uniform_prior(bounds["lb"], bounds["ub"]),
})
params = model.train(batch, random.PRNGKey(0), num_restarts=2)
params.shape


## Score candidates

`score_candidates` evaluates expected improvement in one batched pass. Lower scores are better.


In [ ]:
candidates = jnp.linspace(-2.0, 2.0, 25)[:, None]
scores = acquisitions.score_candidates(
    model, candidates, params=params, batch=batch, bounds=bounds,
    best=float(batch["y"].min()),
)
next_x = candidates[jnp.argmin(scores), 0]
next_y = objective(next_x)
print(f"next point: {float(next_x):.3f}, objective: {float(next_y):.3f}")
assert scores.shape == (25,)
assert jnp.isfinite(scores).all()


The selected point is ready for the next observation in a Bayesian optimization loop.